<img src = "https://drive.google.com/uc?export=view&id=1VV2e_u46fNm_ewns8QW2HGRZAPHh-e2t" alt = "Encabezado MLDS" width = "100%">  </img>

# **Tarea 4 - *Transformers***
---

En esta tarea usted tendrá que usar un _Transformer_ para clasificar un conjunto de noticas de acuerdo a sus temáticas.

> **Importante:  Recomendamos utilizar GPU para la ejecución de este notebook, ya que puede tomar mucho tiempo la ejeución de algunos casos de prueba en caso de que no se utilice.**

# 📘 Tarea 4 · Versión explicada

Copia del notebook resuelto con una explicación antes de cada celda de código. El código y las salidas son los de la ejecución real que obtuvo **100 % en UNCode**, con las 12 pruebas en 100. El original sin explicaciones es `v2_M5U4_Tarea_4.ipynb`.

**Cómo leer cada explicación**
- **📘 Objetivo de la celda:** para qué existe la celda dentro del flujo.
- **Qué hace cada parte del código:** cada instrucción relevante y su función.
- **Punto clave:** el detalle que conviene recordar, cuando lo hay.
- **🔎 Resultado:** la lectura de la salida obtenida.

| | |
|---|---|
| **Problema** | Clasificar noticias de Reuters en 46 temas |
| **Modelo** | Embedding de token y posición → bloque codificador Transformer → fila del token `[CLS]` → `Dense` de 20 → *softmax* de 46 |
| **Datos** | 8.982 noticias de entrenamiento y 2.246 de validación, como listas de índices de palabras |
| **Resultado** | 67,9 % de accuracy de validación; la referencia del enunciado da 68,1 % |

```
Reuters, listas de índices de longitud variable
   → Punto 1 seq_preprocess       [CLS] + recorte y relleno        → (n, 200)
   → Punto 2 get_EmbeddingLayer   token + posición                  → (n, 200, 16)
   → Punto 3 get_EncoderLayer     atención + feed-forward           → (n, 200, 16)
   → Punto 4 get_model            fila del [CLS] → Dense → softmax  → (n, 46)
   → Punto 5 train_model          Adam 1e-4, pérdida dispersa, 10 épocas
```

### 📘 Instalación del calificador

**Objetivo de la celda.** Preparar el entorno para que el notebook pueda enviarse a UNCode.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `%%capture` | Oculta la salida de la celda para no llenar el notebook de mensajes de instalación |
| `!pip install rlxcrypt` | Instala la librería que descifra el módulo del calificador |
| `!wget ... session.pye` | Descarga el módulo cifrado del calificador desde el repositorio del juez de la universidad |

In [1]:
%%capture
!pip install rlxcrypt
!wget --no-cache -O session.pye -q https://raw.githubusercontent.com/JuezUN/INGInious/master/external%20libs/session.pye

### 📘 Inicio de sesión en UNCode

**Objetivo de la celda.** Crear el objeto `grader`, que ejecuta las pruebas y registra la entrega de esta tarea.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `import rlxcrypt, session` | Carga el descifrador y el módulo del calificador |
| `session.LoginSequence("DL-GroupMLDS-5-2026-2@...")` | Abre la sesión del curso y de la tarea identificados en la cadena; pide usuario y contraseña |

> **Punto clave.** Todas las celdas `grader.run_test` y `grader.submit_task` dependen de esta. La contraseña no queda guardada en el notebook.

In [2]:
import rlxcrypt
import session

grader = session.LoginSequence("DL-GroupMLDS-5-2026-2@3feead7a-0040-47fc-b701-11492b6ea6c8")

Please enter your UNCode username: nremolina
Please enter your password: ··········


> Al finalizar, podrás revisar en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/3feead7a-0040-47fc-b701-11492b6ea6c8)

### 📘 Librerías y semillas

**Objetivo de la celda.** Importar las herramientas del notebook y fijar el azar para que los resultados sean reproducibles.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `import keras` | Biblioteca para construir y entrenar la red |
| `import numpy as np` | Arreglos numéricos, usados en el preprocesamiento |
| `import matplotlib.pyplot as plt` | Gráficas; queda disponible aunque la tarea no grafica |
| `random.seed(0)` | Fija el azar de Python |
| `keras.utils.set_random_seed(0)` | Fija el azar de Keras: pesos iniciales, dropout y barajado |
| `np.random.seed(0)` | Fija el azar de NumPy |

> **Punto clave.** Sin semillas, los pesos iniciales cambiarían en cada ejecución y las salidas no podrían compararse con las del enunciado.

In [3]:
import keras
import numpy as np
import matplotlib.pyplot as plt
import os, random
%matplotlib inline
plt.style.use("ggplot")
# Seleccionamos las semillas para efectos de reproducibilidad
random.seed(0)
keras.utils.set_random_seed(0)
np.random.seed(0)

### 📘 Versiones del entorno

**Objetivo de la celda.** Registrar con qué versiones se ejecutó el notebook, para comparar con las del enunciado.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `!python --version` | Muestra la versión de Python |
| `np.__version__, keras.__version__` | Muestran las versiones de NumPy y Keras |

In [4]:
# Versiones de las librerías usadas.
!python --version
print('NumPy', np.__version__)
print('keras', keras.__version__)

Python 3.13.16
NumPy 2.1.3
keras 3.13.2


#### 🔎 Resultado

Python 3.13.16, NumPy 2.1.3 y Keras 3.13.2, más nuevas que las del enunciado: Python 3.11.12, NumPy 2.0.2 y Keras 3.8.0. Todas las pruebas pasaron igual; solo cambian algunos decimales del entrenamiento.

Esta actividad se realizó con las siguientes versiones:
* Python 3.11.12
* NumPy 2.0.2
* keras 3.8.0

## **Cargar Datos**
---

En este caso utilizaremos el dataset [Reuters newswire](https://keras.io/api/datasets/reuters/), se trata de un conjunto de datos de noticas que contiene 11,228 noticias de Reuters, etiquetadas en 46 temas diferentes. Cada noticia está codificada como una lista de índices de palabras, donde las palabras están indexadas por su frecuencia general.

### 📘 Carga del dataset Reuters

**Objetivo de la celda.** Obtener las noticias, sus etiquetas y el diccionario de palabras, con los ids 0 y 1 libres para el relleno y el `[CLS]`.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `vocab_size = 20000` | Limita el vocabulario a las 20.000 palabras más frecuentes |
| `reuters.load_data(num_words=vocab_size, ...)` | Descarga las noticias ya divididas en entrenamiento y validación |
| `index_from=2` | Desplaza los índices de las palabras dos lugares: el 0 queda para el relleno y el 1 para el `[CLS]` |
| `start_char=None` | Evita que Keras agregue su propio token de inicio, porque el `[CLS]` se añade en el punto 1 |
| `get_word_index(...)` | Descarga el diccionario palabra → índice |
| `{valor: clave for clave, valor in words.items()}` | Invierte el diccionario a índice → palabra, para poder leer una noticia |

In [5]:
vocab_size = 20000

(x_train, y_train), (x_val, y_val) = keras.datasets.reuters.load_data(num_words=vocab_size, index_from=2, path=0, start_char = None)
words = keras.datasets.reuters.get_word_index(path="reuters_word_index.json")
words = {valor: clave for clave, valor in words.items()}
print(len(x_train), "Training sequences")
print(len(x_val), "Validation sequences")

2110848/2110848 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
550378/550378 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
8982 Training sequences
2246 Validation sequences


#### 🔎 Resultado

8.982 noticias de entrenamiento y 2.246 de validación, como indica el enunciado.

veamos un ejemeplo:

### 📘 Ver una noticia

**Objetivo de la celda.** Inspeccionar cómo luce una noticia reconstruyendo su texto a partir de los índices.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `for i in x_train[0]` | Recorre los índices de la primera noticia de entrenamiento |
| `words[i]` | Busca la palabra de cada índice en el diccionario invertido |
| `' '.join(...)` | Une las palabras con espacios |

In [6]:
print(f"texto: {' '.join([words[i] for i in x_train[0]])}")

texto: of of a not 3 move in by should 22 in rebates dollar 000 reuters four after about and may in on february said on some reuter after about revs that secretary at and which to but a right would sale 31 said end said been for reuter that earlier for reuter and which mln representation improved noted said domestic said high for reuter that under loss for reuter 000 a sources versus after about last with sale 2 was 12 said co reuter 1 vs


#### 🔎 Resultado

El texto sale desordenado. No es un error de la solución: con `index_from=2` cada índice es el original más 2, y esta celda de la plantilla no resta ese desplazamiento. Para leer la noticia real habría que usar `words.get(i - 2)`. La celda es solo de inspección.

# **1. Normalización de secuencias**
---
A continuación, deberá normalizar las secuencias de manera tal que todas tengan la misma longitud. Recuerde añadir el token `[CLS]`.

**Entradas**:

* **`X`**: `list`, lista que contiene cada una de las secuencias de tamaño variable.
* **`cls_id`**: `int`, ID utilizado para identificar el token `[CLS]`.
* **`max_len`**: `int`, Tamaño final de las sequencias.

**Salida**:

* **`X_out`**:  `arreglo numpy`, arreglo de tamaño (n, m), donde **n** corresponde al número de filas y **m** al tamaño de la secuencia `max_len`.

**Consideraciones importantes**:

* Recuerde considerar el token `[CLS]` en el tamaño de las sequencias.

* Tenga presente que el _relleno_ debe usarse para completar la secuencia en la parte FINAL.

### 📘 Punto 1 · seq_preprocess

**Objetivo de la celda.** Llevar todas las noticias a 200 posiciones, con el `[CLS]` al inicio, relleno al final y, en las largas, conservando el final de la noticia.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `pad_sequences(X, maxlen=max_len - 1, ...)` | Deja cada noticia en 199 posiciones; reserva un lugar para el `[CLS]` |
| `padding='post'` | Completa con ceros al final las noticias cortas, como exige el enunciado |
| `truncating por defecto, 'pre'` | En las noticias largas descarta el inicio y conserva el final; es lo que espera el calificador |
| `value=0` | Usa el id 0, que no corresponde a ninguna palabra, como relleno |
| `np.full((n, 1), cls_id, dtype=X_pad.dtype)` | Crea una columna con el id del `[CLS]` para todas las filas, del mismo tipo de dato |
| `np.concatenate([cls_col, X_pad], axis=1)` | Pega esa columna al frente: cada fila queda `[1, w₁, …, 0, 0]` |

> **Punto clave.** El `[CLS]` se antepone **después** de recortar. La primera versión lo anteponía antes y recortaba por el final: obtuvo 50 en las pruebas 1_2 y 1_3. Con el `[CLS]` ya puesto, un recorte por el inicio lo habría eliminado.

In [11]:
# FUNCIÓN CALIFICADA seq_preprocess
def seq_preprocess(X, cls_id, max_len):
    # Se rellena/trunca a max_len - 1 (espacio para [CLS]); relleno al final
    X_pad = keras.preprocessing.sequence.pad_sequences(
        X, maxlen=max_len - 1, padding='post', value=0
    )
    cls_col = np.full((X_pad.shape[0], 1), cls_id, dtype=X_pad.dtype)
    X_out = np.concatenate([cls_col, X_pad], axis=1)
    return X_out

### 📘 Prueba local del punto 1

**Objetivo de la celda.** Aplicar `seq_preprocess` a ambos conjuntos y comprobar que la forma es la esperada.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `max_len = 200` | Longitud final de cada secuencia |
| `seq_preprocess(x_train, 1, ...)` | Preprocesa el entrenamiento con `cls_id = 1` |
| `seq_preprocess(x_val, 1, ...)` | Preprocesa la validación con los mismos parámetros |
| `print(... .shape)` | Muestra las dimensiones de ambos arreglos |

In [12]:
# TEST CELL
max_len = 200

x_train_p = seq_preprocess(x_train, 1, max_len = max_len)
x_val_p = seq_preprocess(x_val, 1, max_len = max_len)

print(f'Tamaño de la particion de entrenamiento:{x_train_p.shape}')
print(f'Tamaño de la particion de validacion:{x_val_p.shape}')

Tamaño de la particion de entrenamiento:(8982, 200)
Tamaño de la particion de validacion:(2246, 200)


#### 🔎 Resultado

`(8982, 200)` y `(2246, 200)`: coincide con la salida esperada.

**Salida esperada:**
```
Tamaño de la particion de entrenamiento:(8982, 200)
Tamaño de la particion de validacion:(2246, 200)
```




### **Evaluar código**

### 📘 Pruebas del punto 1

**Objetivo de la celda.** Verificar con el calificador la forma y el contenido de `seq_preprocess`. Esta explicación cubre las tres celdas siguientes.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `grader.run_test("Test 1_1", ...)` | Comprueba la forma y los casos de noticias cortas |
| `grader.run_test("Test 1_2", ...)` | Compara el contenido exacto, incluidas noticias de más de 199 palabras |
| `grader.run_test("Test 1_3", ...)` | Otra comparación de contenido exacto con casos largos |

> **Punto clave.** Las pruebas 1_2 y 1_3 son las que detectan de qué lado se recorta.

In [13]:
grader.run_test("Test 1_1", globals())

Test 1_1


In [14]:
grader.run_test("Test 1_2", globals())

Test 1_2


In [15]:
grader.run_test("Test 1_3", globals())

Test 1_3


# **2. Capas de codificación**
----
Cree una capa personalizada de _Keras_ que le permita obtener la representación de cada token junto a la codificación posicional.

**Entrada**:

* **`max_len`**: `int`, Tamaño final de las sequencias.
* **`vocab_size`**: `int`, Cantidad de palabras en el vocabulario.
* **`embed_dim`**: `int`, Tamaño de la representación obtenida por cada token.

**Salida**:
* **`layer`**:  `keras.layer.layer`, Capa personalizada de _Keras_.


### 📘 Punto 2 · get_EmbeddingLayer

**Objetivo de la celda.** Definir una capa propia que represente cada token como la suma de su embedding de palabra y su embedding de posición.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `class TokenAndPositionEmbedding(keras.layers.Layer)` | Declara la capa personalizada heredando de la clase base de Keras |
| `super().__init__()` | Inicializa la capa base para que Keras la registre |
| `self.token_emb = Embedding(vocab_size, embed_dim)` | Tabla de 20.000 × `embed_dim`: un vector por palabra |
| `self.pos_emb = Embedding(max_len, embed_dim)` | Tabla de 200 × `embed_dim`: un vector por posición |
| `seq_len = keras.ops.shape(x)[-1]` | Lee la longitud de la secuencia de entrada |
| `keras.ops.arange(0, seq_len, 1)` | Genera las posiciones `[0, 1, …, 199]` |
| `self.pos_emb(positions)` | Convierte cada posición en su vector |
| `self.token_emb(x)` | Convierte cada id de palabra en su vector |
| `return x + positions` | Suma ambos: el resultado dice qué palabra es y dónde está |
| `layer = TokenAndPositionEmbedding(...)` | Crea la instancia que devuelve la función |

> **Punto clave.** La atención no conoce el orden de las palabras; el embedding de posición se lo aporta.

In [16]:
# FUNCIÓN CALIFICADA get_EmbeddingLayer
def get_EmbeddingLayer(max_len, vocab_size, embed_dim):

    class TokenAndPositionEmbedding(keras.layers.Layer):
        def __init__(self, max_len, vocab_size, embed_dim):
            super().__init__()
            self.token_emb = keras.layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
            self.pos_emb = keras.layers.Embedding(input_dim=max_len, output_dim=embed_dim)

        def call(self, x):
            seq_len = keras.ops.shape(x)[-1]
            positions = keras.ops.arange(0, seq_len, 1)
            positions = self.pos_emb(positions)
            x = self.token_emb(x)
            return x + positions

    layer = TokenAndPositionEmbedding(max_len, vocab_size, embed_dim)
    return layer

### 📘 Prueba local del punto 2

**Objetivo de la celda.** Comprobar que la capa transforma un lote de noticias en la forma esperada.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `get_EmbeddingLayer(max_len, vocab_size, 128)` | Crea la capa con vectores de 128 dimensiones |
| `emb_layer(x_train_p[:10])` | Aplica la capa a las 10 primeras noticias preprocesadas |
| `print(codified_x.shape)` | Muestra la forma del resultado |

In [17]:
# TEST_CELL
max_len = 200
emb_layer = get_EmbeddingLayer(max_len, vocab_size, 128)
codified_x = emb_layer(x_train_p[:10])
print(f'Codificación de sequencias de entrenamiento: {codified_x.shape}')

Codificación de sequencias de entrenamiento: (10, 200, 128)


#### 🔎 Resultado

`(10, 200, 128)`: 10 noticias, 200 posiciones y un vector de 128 números por posición.

**Salida esperada**:
```
Codificación de sequencias de entrenamiento: (10, 200, 128)

```

### **Evaluar código**

### 📘 Pruebas del punto 2

**Objetivo de la celda.** Verificar con el calificador la capa de embedding. Esta explicación cubre las tres celdas siguientes, `Test 2_1` a `Test 2_3`.

> **Punto clave.** Las tres dieron 100 en el primer intento.

In [18]:
grader.run_test("Test 2_1", globals())

Test 2_1


In [19]:
grader.run_test("Test 2_2", globals())

Test 2_2


In [21]:
grader.run_test("Test 2_3", globals())

Test 2_3


# **3. Codificador**
----
Construya una capa personalizada de Keras que genere el codificador de un Transformer

**Entrada**:

* **`embed_dim`**:`int`, Tamaño de la representación de cada token.
* **`num_heads`**:`int`,  Número de cabezas de atención
* **`ff_dim`**:`int`, Tamaño de la representación final obtenida por el codificador.
* **`rate`**:`float`, Tasa de apagado para la capa de `Dropout`.

**Salida**:
* **`layer`**:  `keras.layer.layer`, Capa personalizada de _Keras_.

### 📘 Punto 3 · get_EncoderLayer

**Objetivo de la celda.** Definir el bloque codificador del Transformer: cada palabra se reinterpreta según las demás y luego se transforma por posición.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `self.att = MultiHeadAttention(num_heads, key_dim=embed_dim)` | Atención multicabeza: cada token calcula cuánto mirar a cada otro |
| `self.ffn = Sequential([Dense(ff_dim, relu), Dense(embed_dim)])` | Red de dos capas aplicada a cada posición: expande y vuelve al tamaño original |
| `LayerNormalization(epsilon=1e-6)` | Normaliza cada vector; `epsilon` evita divisiones por cero |
| `Dropout(rate)` | Apaga al azar una fracción de valores durante el entrenamiento |
| `call(self, inputs, training=False)` | Recibe la secuencia y si se está entrenando; Keras pasa `training=True` durante `fit` |
| `self.att(inputs, inputs)` | Auto-atención: la secuencia se mira a sí misma |
| `self.layernorm1(inputs + attn_output)` | Suma residual y normalización del primer subbloque |
| `self.ffn(out1)` | Transforma cada posición de forma no lineal |
| `self.layernorm2(out1 + ffn_output)` | Suma residual y normalización del segundo subbloque |

> **Punto clave.** La salida tiene la misma forma que la entrada, `(lote, 200, embed_dim)`. Las sumas residuales dejan pasar la información original y facilitan el entrenamiento.

In [22]:
# FUNCIÓN CALIFICADA get_EncoderLayer
def get_EncoderLayer(embed_dim, num_heads, ff_dim, rate):
    class TransformerBlock(keras.layers.Layer):
        def __init__(self, embed_dim, num_heads, ff_dim, rate):
            super().__init__()
            self.att = keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
            self.ffn = keras.Sequential([
                keras.layers.Dense(ff_dim, activation="relu"),
                keras.layers.Dense(embed_dim),
            ])
            self.layernorm1 = keras.layers.LayerNormalization(epsilon=1e-6)
            self.layernorm2 = keras.layers.LayerNormalization(epsilon=1e-6)
            self.dropout1 = keras.layers.Dropout(rate)
            self.dropout2 = keras.layers.Dropout(rate)

        def call(self, inputs, training=False):
            attn_output = self.att(inputs, inputs)
            attn_output = self.dropout1(attn_output, training=training)
            out1 = self.layernorm1(inputs + attn_output)
            ffn_output = self.ffn(out1)
            ffn_output = self.dropout2(ffn_output, training=training)
            return self.layernorm2(out1 + ffn_output)

    layer = TransformerBlock(embed_dim, num_heads, ff_dim, rate)
    return layer

### 📘 Prueba local del punto 3

**Objetivo de la celda.** Comprobar que el bloque conserva la forma de su entrada.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `get_EncoderLayer(128, 2, 32, .1)` | Crea el bloque con vectores de 128, 2 cabezas, red interna de 32 y dropout de 0,1 |
| `encoder_layer(codified_x)` | Aplica el bloque a la salida de la capa de embedding |

In [23]:
# TEST_CELL
encoder_layer = get_EncoderLayer(128, 2, 32, .1)
encoded_x = encoder_layer(codified_x)
print(f'Codificación de sequencias de entrenamiento: {encoded_x.shape}')

Codificación de sequencias de entrenamiento: (10, 200, 128)


#### 🔎 Resultado

`(10, 200, 128)`: la forma no cambia, pero cada vector ya incorpora información del resto de la noticia.

**Salida esperada**
```
Codificación de sequencias de entrenamiento: (10, 200, 128)
```

### **Evaluar código**

### 📘 Pruebas del punto 3

**Objetivo de la celda.** Verificar con el calificador el bloque codificador. Esta explicación cubre las dos celdas siguientes, `Test 3_1` y `Test 3_2`.

> **Punto clave.** Las dos dieron 100 en el primer intento.

In [24]:
grader.run_test("Test 3_1", globals())

Test 3_1


In [25]:
grader.run_test("Test 3_2", globals())

Test 3_2


# **4. Construir modelo**
----
Complete la función `get_model` que construye el modelo a partir de las estructuras diseñadas anteriormente. Recuerde utilizar el token `[CLS]` para obtener la representación final después del codificador. Adicionalmente, tenga en cuenta que el modelo debe desempeñarse para una tarea de clasificación multiclase, por lo que debe emplear la activación, función de pérdida y métrica adecuada para este contexto.


**Entrada**:

* **`max_len`**: `int`, Tamaño final de las sequencias.
* **`vocab_size`**: `int`, Cantidad de palabras en el vocabulario.
* **`embed_dim`**:`int`, Tamaño de la representación de cada token.
* **`num_heads`**:`int`,  Número de cabezas de atención
* **`ff_dim`**:`int`, Tamaño de la representación final obtenida por el codificador.
* **`rate`**:`float`, Tasa de apagado para la capa de `Dropout`.
* **`num_clases`**:`int`,  Número de clases de salida.
* **`learning_rate`**:`float`, Tasa de aprendizaje.
* **`n_neurons`**:`int`, Número de neuronas en la capa intermedia del perceptron multicapa para la clasificación.

**Salida**:
* **`model`**:  `keras.Model`, modelo de _Keras_.


> **Nota**: Debido a la estrutura de las etiquetas determine si debe uilizar la forma dispersa de la función deperdida.

### 📘 Punto 4 · get_model

**Objetivo de la celda.** Ensamblar el modelo completo y compilarlo para clasificación multiclase.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `np.random.seed(0), set_random_seed(0), clear_session()` | Venían dadas: reinician Keras para que el modelo salga idéntico en cada llamada |
| `keras.layers.Input(shape=(max_len,))` | Declara la entrada: secuencias de 200 enteros |
| `get_EmbeddingLayer(...)(inputs)` | Aplica la capa del punto 2 |
| `get_EncoderLayer(...)(x)` | Aplica el bloque del punto 3 |
| `x[:, 0, :]` | Toma solo la fila del `[CLS]`, el resumen de la noticia |
| `Dense(n_neurons, activation="relu")` | Capa intermedia de 20 neuronas |
| `Dense(num_clases, activation="softmax")` | Una probabilidad por cada uno de los 46 temas |
| `keras.Model(inputs, outputs)` | Registra el grafo completo con la API funcional |
| `Adam(learning_rate=learning_rate)` | Optimizador con la tasa de aprendizaje recibida |
| `loss="sparse_categorical_crossentropy"` | Pérdida para etiquetas enteras; es la «forma dispersa» que pide la nota |
| `metrics=["accuracy"]` | Proporción de noticias bien clasificadas |

> **Punto clave.** El `[CLS]` no es una palabra: gracias a la atención acumuló información de toda la noticia, y el clasificador solo lee esa fila.

In [26]:
# FUNCIÓN CALIFICADA get_model
def get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate, num_clases, learning_rate, n_neurons):
    # Fijamos una semilla para efectos de reproducibiidad
    np.random.seed(0)
    keras.utils.set_random_seed(0)
    keras.backend.clear_session()

    inputs = keras.layers.Input(shape=(max_len,))
    x = get_EmbeddingLayer(max_len, vocab_size, embed_dim)(inputs)
    x = get_EncoderLayer(embed_dim, num_heads, ff_dim, rate)(x)
    x = x[:, 0, :]  # representación del token [CLS]
    x = keras.layers.Dense(n_neurons, activation="relu")(x)
    outputs = keras.layers.Dense(num_clases, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

### 📘 Prueba local del punto 4

**Objetivo de la celda.** Construir el modelo con los hiperparámetros de la prueba y comparar su resumen con el esperado.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `embed_dim = 16, num_heads = 2, ff_dim = 32` | Tamaño de los vectores, cabezas de atención y red interna |
| `num_clases = 46, n_neurons = 20` | Clases de salida y neuronas de la capa intermedia |
| `learning_rate = 1e-4, rate = 0.1` | Tasa de aprendizaje y de dropout |
| `model.summary()` | Muestra capas, formas y número de parámetros |

In [27]:
# TEST_CELL
max_len = 200
vocab_size = 20000
embed_dim = 16
num_heads = 2
ff_dim = 32
rate = 0.1
num_clases = 46
learning_rate = 1e-4
n_neurons = 20

model = get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate, num_clases, learning_rate, n_neurons)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 200, 16)        │       323,200 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 200, 16)        │         3,296 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20)             │           340 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 46)             │           966 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 327,802 (1.25 MB)

 Trainable params: 327,802 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

#### 🔎 Resultado

Coincide exactamente con la salida esperada: **327.802 parámetros**.

| Componente | Parámetros |
|---|---:|
| Embedding de palabras, 20.000 × 16 | 320.000 |
| Embedding de posiciones, 200 × 16 | 3.200 |
| Bloque codificador | 3.296 |
| `dense_2`, 16 × 20 + 20 | 340 |
| `dense_3`, 20 × 46 + 46 | 966 |

El 98,6 % está en los embeddings. La capa `get_item`, con 0 parámetros, es el `x[:, 0, :]` del `[CLS]`.

**Salida esperada:** (Últimos renglones)
```
Model: "functional_1"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding    │ (None, 200, 16)        │       323,200 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 200, 16)        │         3,296 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ get_item (GetItem)              │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 20)             │           340 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 46)             │           966 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 327,802 (1.25 MB)
 Trainable params: 327,802 (1.25 MB)
 Non-trainable params: 0 (0.00 B)
```

### 📘 Configuración interna del modelo

**Objetivo de la celda.** Inspeccionar la configuración completa: capas, activaciones y conexiones.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `pprint.pprint(model.get_config())` | Imprime con formato el diccionario de configuración del modelo |

In [28]:
import pprint

pprint.pprint(model.get_config())

{'input_layers': ['input_layer', 0, 0],
 'layers': [{'class_name': 'InputLayer',
             'config': {'batch_shape': (None, 200),
                        'dtype': 'float32',
                        'name': 'input_layer',
                        'optional': False,
                        'ragged': False,
                        'sparse': False},
             'inbound_nodes': [],
             'module': 'keras.layers',
             'name': 'input_layer',
             'registered_name': None},
            {'build_config': {'input_shape': (None, 200)},
             'class_name': 'TokenAndPositionEmbedding',
             'config': {'dtype': {'class_name': 'DTypePolicy',
                                  'config': {'name': 'float32'},
                                  'module': 'keras',
                                  'registered_name': None},
                        'embed_dim': 16,
                        'max_len': 200,
                        'trainable': True,
                      

#### 🔎 Resultado

Difiere de la salida esperada: allí las capas propias muestran sus hiperparámetros porque implementan `get_config()`, y esta versión no lo hace. Las pruebas 4_1 y 4_2 dieron 100 de todos modos. La salida esperada además dice `embed_dim: 32` aunque el modelo usa 16.

**Salida esperada:** (Últimos renglones)
```
{'input_layers': [['input_layer', 0, 0]],
 'layers': [{'class_name': 'InputLayer',
             'config': {'batch_shape': (None, 200),
                        'dtype': 'float32',
                        'name': 'input_layer',
                        'sparse': False},
             'inbound_nodes': [],
             'module': 'keras.layers',
             'name': 'input_layer',
             'registered_name': None},
            {'build_config': {'input_shape': (None, 200)},
             'class_name': 'TokenAndPositionEmbedding',
             'config': {'dtype': {'class_name': 'DTypePolicy',
                                  'config': {'name': 'float32'},
                                  'module': 'keras',
                                  'registered_name': None},
                        'embed_dim': 32,
                        'max_len': 200,
                        'trainable': True,
                        'vocab_size': 20000},
             'inbound_nodes': [{'args': ({'class_name': '__keras_tensor__',
                                          'config': {'dtype': 'float32',
                                                     'keras_history': ['input_layer',
                                                                       0,
                                                                       0],
                                                     'shape': (None, 200)}},),
                                'kwargs': {}}],
             'module': None,
             'name': 'token_and_position_embedding',
             'registered_name': 'TokenAndPositionEmbedding'},
            {'build_config': {'input_shape': (None, 200, 32)},
             'class_name': 'TransformerBlock',
             'config': {'dtype': {'class_name': 'DTypePolicy',
                                  'config': {'name': 'float32'},
                                  'module': 'keras',
                                  'registered_name': None},
                        'embed_dim': 32,
                        'ff_dim': 32,
                        'num_heads': 2,
                        'rate': 0.1,
                        'trainable': True},
             'inbound_nodes': [{'args': ({'class_name': '__keras_tensor__',
                                          'config': {'dtype': 'float32',
                                                     'keras_history': ['token_and_position_embedding',
                                                                       0,
                                                                       0],
                                                     'shape': (None,
                                                               200,
                                                               32)}},),
                                'kwargs': {}}],
             'module': None,
             'name': 'transformer_block',
             'registered_name': 'TransformerBlock'},
            {'build_config': {'input_shape': (None, 200, 32)},
             'class_name': 'Dense',
             'config': {'activation': 'relu',
                        'bias_constraint': None,
                        'bias_initializer': {'class_name': 'Zeros',
                                             'config': {},
                                             'module': 'keras.initializers',
                                             'registered_name': None},
                        'bias_regularizer': None,
                        'dtype': {'class_name': 'DTypePolicy',
                                  'config': {'name': 'float32'},
                                  'module': 'keras',
                                  'registered_name': None},
                        'kernel_constraint': None,
                        'kernel_initializer': {'class_name': 'GlorotUniform',
                                               'config': {'seed': None},
                                               'module': 'keras.initializers',
                                               'registered_name': None},
                        'kernel_regularizer': None,
                        'name': 'dense_2',
                        'trainable': True,
                        'units': 20,
                        'use_bias': True},
             'inbound_nodes': [{'args': ({'class_name': '__keras_tensor__',
                                          'config': {'dtype': 'float32',
                                                     'keras_history': ['transformer_block',
                                                                       0,
                                                                       0],
                                                     'shape': (None,
                                                               200,
                                                               32)}},),
                                'kwargs': {}}],
             'module': 'keras.layers',
             'name': 'dense_2',
             'registered_name': None},
            {'build_config': {'input_shape': (None, 200, 20)},
             'class_name': 'Dense',
             'config': {'activation': 'softmax',
                        'bias_constraint': None,
                        'bias_initializer': {'class_name': 'Zeros',
                                             'config': {},
                                             'module': 'keras.initializers',
                                             'registered_name': None},
                        'bias_regularizer': None,
                        'dtype': {'class_name': 'DTypePolicy',
                                  'config': {'name': 'float32'},
                                  'module': 'keras',
                                  'registered_name': None},
                        'kernel_constraint': None,
                        'kernel_initializer': {'class_name': 'GlorotUniform',
                                               'config': {'seed': None},
                                               'module': 'keras.initializers',
                                               'registered_name': None},
                        'kernel_regularizer': None,
                        'name': 'dense_3',
                        'trainable': True,
                        'units': 46,
                        'use_bias': True},
             'inbound_nodes': [{'args': ({'class_name': '__keras_tensor__',
                                          'config': {'dtype': 'float32',
                                                     'keras_history': ['dense_2',
                                                                       0,
                                                                       0],
                                                     'shape': (None,
                                                               200,
                                                               20)}},),
                                'kwargs': {}}],
             'module': 'keras.layers',
             'name': 'dense_3',
             'registered_name': None}],
 'name': 'functional_1',
 'output_layers': [['dense_3', 0, 0]],
 'trainable': True}
```

### **Evaluar código**

### 📘 Pruebas del punto 4

**Objetivo de la celda.** Verificar con el calificador el modelo ensamblado. Esta explicación cubre las dos celdas siguientes, `Test 4_1` y `Test 4_2`.

> **Punto clave.** Las dos dieron 100 en el primer intento.

In [29]:
grader.run_test("Test 4_1", globals())

Test 4_1


In [30]:
grader.run_test("Test 4_2", globals())

Test 4_2


# **5. Entrene el modelo**
----
Complete la función `train_model` que recibe un modelo y lo ajusta usando las secuencias de entrenamiento.

**Entrada**:

* **`model`**:  `keras.Model`, modelo de _Keras_ compilado.
* **`cls_id`**: `int`, ID utilizado para identificar el token `[CLS]`.
* **`X_train`**:  `list`, lista que contiene cada una de las secuencias de entrenamiento.
* **`Y_train`**:  `array`, Arreglo con las etiquetas de cada observación del conjunto de entrenamiento.
* **`X_val`**:  `list`, lista que contiene cada una de las secuencias de validación.
* **`Y_val`**:  `array`, Arreglo con las etiquetas de cada observación del conjunto de validación.
* **`max_len`**: `int`, Tamaño final de las sequencias.
* **`epochs`**: `int`, Numero de epocas.

**Salida**:
* **`model`**:  `keras.Model`, modelo de _Keras_ compilado y entrenado.
* **`history`**:  Objeto, objeto que contiene las métricas de evaluación y entrenamiento a lo largo de las épocas.

### 📘 Punto 5 · train_model

**Objetivo de la celda.** Preprocesar los datos crudos y entrenar el modelo, midiendo la validación en cada época.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `seq_preprocess(X_train, cls_id, max_len)` | Prepara el entrenamiento con el formato del punto 1 |
| `seq_preprocess(X_val, cls_id, max_len)` | Prepara la validación con exactamente el mismo formato |
| `model.fit(X_train_p, Y_train, ...)` | Entrena; sin `batch_size`, Keras usa 32: 281 pasos por época |
| `validation_data=(X_val_p, Y_val)` | Mide la validación al final de cada época |
| `return history` | Devuelve el registro de métricas por época |

> **Punto clave.** El enunciado dice que la función devuelve el modelo y `history`, pero la celda siguiente recibe un solo valor. El modelo se entrena en el lugar, así que basta `history`.

In [31]:
# FUNCIÓN CALIFICADA train_model
def train_model(model, cls_id, X_train, Y_train, X_val, Y_val, max_len, epochs):
    X_train_p = seq_preprocess(X_train, cls_id, max_len)
    X_val_p = seq_preprocess(X_val, cls_id, max_len)

    history = model.fit(
        X_train_p, Y_train,
        validation_data=(X_val_p, Y_val),
        epochs=epochs,
    )
    return history

### 📘 Entrenamiento de 10 épocas

**Objetivo de la celda.** Construir un modelo nuevo y entrenarlo con la configuración del enunciado.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `get_model(...)` | Crea un modelo desde cero |
| `train_model(model, 1, x_train, y_train, x_val, y_val, 200, 10)` | Lo entrena 10 épocas con `cls_id = 1` y `max_len = 200` |

In [32]:
max_len = 200
epochs = 10
cls_id = 1
model = get_model(max_len, vocab_size, embed_dim, num_heads, ff_dim, rate, num_clases, learning_rate, n_neurons)
history = train_model(model, cls_id, x_train, y_train, x_val, y_val, max_len, epochs)

Epoch 1/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 19s 36ms/step - accuracy: 0.3195 - loss: 2.9635 - val_accuracy: 0.4920 - val_loss: 2.5244
Epoch 2/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4963 - loss: 2.3074 - val_accuracy: 0.5392 - val_loss: 2.0730
Epoch 3/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5532 - loss: 1.8922 - val_accuracy: 0.5748 - val_loss: 1.7553
Epoch 4/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5941 - loss: 1.6550 - val_accuracy: 0.6051 - val_loss: 1.6448
Epoch 5/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6328 - loss: 1.5320 - val_accuracy: 0.6402 - val_loss: 1.5449
Epoch 6/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6650 - loss: 1.4291 - val_accuracy: 0.6532 - val_loss: 1.4920
Epoch 7/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6910 - loss: 1.3291 - val_accuracy: 0.6705 - val_loss: 1.4318
Epoch 8/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7110 - loss: 1.2399 - val_accuracy: 

#### 🔎 Resultado

| Época | Accuracy | Validación | Validación esperada |
|---:|---:|---:|---:|
| 1 | 0,3195 | 0,4920 | 0,4920 |
| 5 | 0,6328 | 0,6402 | 0,6402 |
| 10 | 0,7341 | **0,6794** | 0,6808 |

- **Aprendizaje sostenido:** entrenamiento y validación suben juntos; la brecha final es de unos 5,5 puntos.
- **Frente a las líneas base:** el azar daría cerca de 2 % y predecir siempre el tema más frecuente, cerca de 36 %.
- **Frente a la referencia:** las cinco primeras épocas de validación son idénticas; las diferencias posteriores son de milésimas, por la GPU y la versión de Keras.

**Salida esperada**
```
Epoch 1/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 14s 28ms/step - accuracy: 0.1947 - loss: 3.2666 - val_accuracy: 0.4920 - val_loss: 2.5244
Epoch 2/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.4833 - loss: 2.4337 - val_accuracy: 0.5392 - val_loss: 2.0730
Epoch 3/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5420 - loss: 1.9843 - val_accuracy: 0.5748 - val_loss: 1.7553
Epoch 4/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5842 - loss: 1.7040 - val_accuracy: 0.6051 - val_loss: 1.6448
Epoch 5/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6195 - loss: 1.5705 - val_accuracy: 0.6402 - val_loss: 1.5449
Epoch 6/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6535 - loss: 1.4593 - val_accuracy: 0.6523 - val_loss: 1.4933
Epoch 7/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.6784 - loss: 1.3613 - val_accuracy: 0.6701 - val_loss: 1.4317
Epoch 8/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7022 - loss: 1.2662 - val_accuracy: 0.6705 - val_loss: 1.3829
Epoch 9/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.7146 - loss: 1.1924 - val_accuracy: 0.6763 - val_loss: 1.3511
Epoch 10/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7228 - loss: 1.1215 - val_accuracy: 0.6808 - val_loss: 1.3179
```

### **Evaluar código**

### 📘 Pruebas del punto 5

**Objetivo de la celda.** Verificar con el calificador el entrenamiento. Esta explicación cubre las dos celdas siguientes, `Test 5_1` y `Test 5_2`.

> **Punto clave.** Son las pruebas más lentas: cada una construye y entrena un modelo desde cero con estas funciones, por eso muestran épocas.

In [33]:
grader.run_test("Test 5_1", globals())

Test 5_1
281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.3195 - loss: 2.9635 - val_accuracy: 0.4920 - val_loss: 2.5244


In [34]:
grader.run_test("Test 5_2", globals())

Test 5_2
Epoch 1/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.2175 - loss: 3.0986 - val_accuracy: 0.4230 - val_loss: 2.5847
Epoch 2/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4134 - loss: 2.4329 - val_accuracy: 0.5000 - val_loss: 2.2453
Epoch 3/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5082 - loss: 2.0951 - val_accuracy: 0.5583 - val_loss: 1.8879
Epoch 4/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5677 - loss: 1.7808 - val_accuracy: 0.5744 - val_loss: 1.7292
Epoch 5/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.5902 - loss: 1.6292 - val_accuracy: 0.6046 - val_loss: 1.6187
Epoch 6/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.6276 - loss: 1.5279 - val_accuracy: 0.6336 - val_loss: 1.5819
Epoch 7/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6662 - loss: 1.4331 - val_accuracy: 0.6483 - val_loss: 1.5258
Epoch 8/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6922 - loss: 1.3382 - val_a

# **Evaluación**


### 📘 Envío oficial

**Objetivo de la celda.** Registrar la entrega en UNCode.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `grader.submit_task(globals())` | Vuelve a ejecutar las 12 pruebas en el servidor y guarda la calificación |

In [35]:
grader.submit_task(globals())

Test 1_1
Test 1_2
Test 1_3
Test 2_1
Test 2_2
Test 2_3
Test 3_1
Test 3_2
Test 4_1
Test 4_2
Test 5_1
281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.3195 - loss: 2.9635 - val_accuracy: 0.4920 - val_loss: 2.5244
Test 5_2
Epoch 1/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 12s 22ms/step - accuracy: 0.2175 - loss: 3.0986 - val_accuracy: 0.4230 - val_loss: 2.5847
Epoch 2/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4134 - loss: 2.4329 - val_accuracy: 0.5000 - val_loss: 2.2453
Epoch 3/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5082 - loss: 2.0951 - val_accuracy: 0.5583 - val_loss: 1.8879
Epoch 4/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5677 - loss: 1.7808 - val_accuracy: 0.5744 - val_loss: 1.7292
Epoch 5/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5902 - loss: 1.6292 - val_accuracy: 0.6046 - val_loss: 1.6187
Epoch 6/10
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6276 - loss: 1.5279 - val_accuracy: 0.6336 - val_loss: 1.5819
Epo

#### 🔎 Resultado

**Las 12 pruebas en 100 y «Your submission grade: 100.0%».** Hubo un envío anterior con 1_2 y 1_3 en 50, antes de corregir el recorte.

Revisa en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/3feead7a-0040-47fc-b701-11492b6ea6c8)

# 📘 Cierre

| Punto | Función | Resultado | Qué fue decisivo |
|---|---|---|---|
| 1 | `seq_preprocess` | 100, 100, 100 | Conservar el final de las noticias largas y anteponer el `[CLS]` después de recortar |
| 2 | `get_EmbeddingLayer` | 100, 100, 100 | Sumar embedding de palabra y de posición |
| 3 | `get_EncoderLayer` | 100, 100 | Atención, residuales, normalización y dropout en el orden estándar |
| 4 | `get_model` | 100, 100 | Leer solo la fila del `[CLS]` y usar pérdida dispersa |
| 5 | `train_model` | 100, 100 | Preprocesar entrenamiento y validación con el mismo formato |

**Ideas para recordar**
1. El `[CLS]` es un resumen aprendido de la noticia.
2. La posición hay que sumarla, porque la atención no conoce el orden.
3. Etiquetas enteras implican pérdida dispersa.
4. Qué extremo se recorta es una decisión de diseño con consecuencias.

# **Créditos**
---

* **Profesor:** [Fabio Augusto Gonzalez](https://dis.unal.edu.co/~fgonza/)
* **Asistentes docentes :**
  * [Juan Sebastián Malagón Torres](https://http://juselara.com/)
* **Coordinador de virtualización:**
    - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - *Facultad de Ingeniería*